# Are we finding more Earth-like planets, or just better telescopes?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2FT4_more_planets_or_better_telescopes.ipynb).

The archive of confirmed planets around other stars holds 6,354 of them. Sort those by the
year they were announced and take the middle radius of each year, and the series falls off a cliff:
the typical planet found up to 2010 was 13.0 times the radius of the Earth, and the
typical planet found since 2016 is 2.47.

Read one way, that is the best news in astronomy. We went looking for other Earths and we started
finding them, and the number of small planets in the file has been climbing ever since.

Read the other way, nothing about the planets changed at all. Kepler launched in 2009 and stared at
one patch of sky for four years; TESS launched in 2018 and has swept the whole sky for bright
nearby stars. Each of them could see a particular kind of planet and was nearly blind to the rest.
What fell in 2013 might be the size of the planets we were finding, or it might only be the size of
the planets we had become able to find. This project is about telling those two apart — and then
about what is left over once you have.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## How this notebook is different

This is a **project track**. It is not a weekly notebook and it does not behave like one.

A weekly notebook shows you a move, walks you through it, and then asks you to make it once
yourself. This one loads the data and reproduces the single result its title rests on — the fall in
the typical planet radius — and then stops helping. From there on every section is a sentence
describing what to find out and an empty cell to find it out in. There is no worked example above
to pattern-match against, because on a real question there never is one.

**There is exactly one self-check in this notebook, and it is on the data loading.** After that,
nothing tells you whether you are right. That is not an oversight and it is not laziness: past the
loading step there is no single right answer here, so a cell that said `assert` would be lying to
you about how research works. What replaces it is the thing researchers actually use — a result you
can get two ways, a number you can predict before you compute it, and a claim you can try to break.

**And it does not close.** The last section is a question this course does not know the answer to.
Everything above it is scaffolding; that question is the project.

## What you'll be able to do

**The science.** Say whether the shrinking of the typical known exoplanet is a fact about planets
or a fact about telescopes, and defend the answer with a number rather than an adjective. Say which
planets two instruments can honestly be compared on, and which comparisons are between a
measurement and the output of somebody's model. Then correct a survey for one of its biases, and
find out what correcting one bias out of several actually buys you.

**The skills.** Tell a measured column from a modelled one in a real archive, using nothing but the
error bars and a provenance column. Fit a straight line in log space and then use it on data it was
never fitted to, deliberately, to see what that does. Turn a piece of orbital geometry into a
weight, and reweight a sample by it.

**The four questions, in order:**

1. Has the typical newly found planet been getting smaller?
2. Which planets can you honestly compare?
3. Which telescope found them?
4. What would we have seen if geometry had not chosen for us?

The open question at the end is not on that list. It is the project; the four above are what you
build to reach it.

## Setup

The NASA Exoplanet Archive publishes the confirmed-planet catalogue over a plain URL, with no key
and no login. The cell below reads it live and falls back to the copy stored with the course.

**Read this before you go on — it is the whole project in miniature.** The archive serves *two*
tables that both hold one row per planet, and today they both hold **6,354** rows. They are
not the same data:

- **`ps` with `default_flag=1`** is the literature. One row per planet, taken from whichever
  published paper the archive considers the default reference, and a **hole wherever nobody has
  measured that quantity.** Only 74.6% of these planets have a radius.
- **`pscomppars`** is the *composite* table. Same planets, but the holes are filled in — with
  values derived from other columns through a model. 99.2% of these planets
  have a radius.

Everything in this notebook is measured on `ps`. `pscomppars` is loaded as well, and it is used
exactly once, in *Your turn 2*, to show you what the difference between the two is made of.

**NaN:** Where the file had nothing at all, pandas puts NaN. A NaN is a hole, not a zero. That is the whole distinction above, and a comparison like
`radius < 1.6` is `False` for a hole, silently, with no error — so a planet nobody has measured
gets quietly counted as "not rocky".

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load(url, cache_name):
    """Read the live archive; fall back to the copy stored with the course."""
    try:
        return pd.read_csv(url)
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + cache_name)

ARCHIVE = ("https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+"
           "pl_name,pl_rade,pl_radeerr1,pl_bmasse,pl_bmassprov,pl_orbsmax,st_rad,sy_dist,"
           "disc_year,discoverymethod,disc_facility+from+")

planets = load(ARCHIVE + "ps+where+default_flag=1&format=csv", "trackT4_ps.csv")
composite = load(ARCHIVE + "pscomppars&format=csv", "trackT4_pscomppars.csv")

print("ps, the literature: ", planets.shape)
print("pscomppars, filled: ", composite.shape)
print(planets[["pl_name", "pl_rade", "pl_bmasse", "pl_bmassprov", "discoverymethod"]].head())

In [ ]:
assert "pl_bmassprov" in planets.columns, \
    "the mass-provenance column is missing — the query was read wrong, or the schema changed"
assert 6000 < len(planets) < 7500, \
    "expected about 6354 planets; 40,000 means default_flag=1 is missing from the query"
assert planets["pl_name"].duplicated().sum() == 0, \
    "duplicate planet names — without default_flag=1 this is one row per published paper"
print(f"✓ the data — {len(planets)} planets from ps and {len(composite)} from pscomppars, "
      f"{planets['pl_rade'].count()} of the ps rows carrying a measured radius")

### And that is the last self-check in this notebook

The pipeline is now trustworthy: the table is the table, the query is the query, the counts below
are the counts. Everything from here is yours, and the safety net is gone — nothing will tell you
when you have it right.

## Has the typical newly found planet been getting smaller?

One point per year: the middle radius of every planet announced that year, in radii of the Earth.
Only the 4,742 planets that have a measured radius can be on this plot at all; the
other 1,612 are holes, and `median()` steps over them without saying so.

**Catalogue completeness:** A catalogue lists what somebody's instruments recorded, not what happened. Where there are no seismometers there are no earthquakes in the file. That sentence was
written about earthquakes. Read it again with "telescopes" in it and it is this project's whole
argument.

The archive is revised continuously, so your counts may differ from the ones printed in this
notebook by a few. Say so if they do — a record that changes under you is the subject here, not a
nuisance.

In [ ]:
with_radius = planets[planets["pl_rade"].notna()]
per_year = with_radius.groupby("disc_year")["pl_rade"].median()
count_per_year = with_radius.groupby("disc_year")["pl_rade"].count()

plt.plot(per_year.index, per_year.values, marker="o", color="0.3")
plt.xlabel("year the planet was announced")
plt.ylabel("median radius of that year's planets (Earth radii)")
plt.title(f"Median measured planet radius by year (n = {len(with_radius)})")
plt.show()

print(per_year.round(2).to_string())
print("planets per year:", count_per_year.to_dict())

old_planets = with_radius[with_radius["disc_year"] <= 2010]["pl_rade"].values
new_planets = with_radius[with_radius["disc_year"] >= 2016]["pl_rade"].values
print("up to 2010: ", len(old_planets), "planets, median radius",
      round(np.median(old_planets), 2))
print("since 2016: ", len(new_planets), "planets, median radius",
      round(np.median(new_planets), 2))
print("spread of the yearly medians since 2014:",
      round(per_year.loc[2014:].std(), 2), "Earth radii, largest minus smallest",
      round(per_year.loc[2014:].max() - per_year.loc[2014:].min(), 2))

That is the observation the project exists to explain, and it needs no statistics to see: the
middle planet fell from about 13 Earth radii to about 2.5, a factor
of 5.3, and the whole of it happened between 2010 and 2014.

Look at the line after 2014 as well, because a second thing is going on. The yearly median does not
settle — it saws up and down by whole Earth radii from one year to the next
(2015 → 2.62, 2016 → 2.14, 2017 → 3.39, 2025 → 9.53), a swing of 7.4
across 2014–2026 with a standard deviation of 1.93. Nothing
about the sky changes that fast. What changes is which team published a batch that year.

So a *year* is not a useful unit for this data. The first question is what happens when you stop
using it.

One detail in the cell above, because you will copy the shape: the two eras were pulled out with
`.values`, which hands back a plain array instead of a column of the table. Everything below that
resamples or takes a logarithm wants an array, so that is the form this notebook uses whenever a
set of numbers is about to be worked on rather than looked up.

### ✏️ Your turn 1

Compare two eras instead of 13 years: **2016–2020**
against **2021–2026**. Print how many planets with a measured radius each era holds, the
median radius of each, and the difference between the two medians.

Then put an interval on that difference, because a difference with no interval is not a result.

**Bootstrap:** Ask the data the same question a thousand times, using a different random slice of itself each time.
**Confidence interval:** Not one number but the range your number would have wandered over, had the world rolled differently.

Resample each era **with replacement** to its own size, take the difference of the two medians, and
do that 2000 times. `rng = np.random.default_rng(88)` once, then
`rng.choice(values, size=len(values), replace=True)` on each era inside the loop. Report the 2.5th
and 97.5th percentiles, and draw the 2000 differences as a histogram with the observed
difference marked.

Now answer it, in a line your code prints, on your own three numbers — the difference, its
interval, and the year-to-year standard deviation of 1.93 printed above: is the
later era's larger typical planet something you could have seen in the yearly series, and what
would that series have had to look like for the answer to be yes?

In [ ]:
# ← your answer here



## Which planets can you honestly compare?

Every number above is a **radius**, and there is a reason the archive has so many holes in that
column. A transit measures a radius directly: the planet crosses the star, the star dims, and the
fraction it dims by is the ratio of the two areas. Radial velocity measures nothing of the sort. It
watches the star wobble along the line of sight, and the wobble depends on the planet's mass *and*
on how the orbit happens to be tilted — which is unknown. So what comes out is not a mass but a
**minimum** mass, `M·sin(i)`, and no radius at all.

The archive records which one you are looking at, in `pl_bmassprov`: `Mass` means somebody pinned
the true mass, `Msini` means the tilt is unknown and the number is a floor.

**Table:** A table with a name on every column, so you ask for data by name instead of by position.

### Predict before you run

1,200 of the planets in `ps` were discovered by radial velocity. Change `my_guess` to how
many of those you think carry a **measured** radius, and run the cell. You will check it in the
next question, and a wrong guess you committed to is worth more than a right answer you were shown.

In [ ]:
my_guess = 600

print("I think", my_guess, "of the 1200 radial-velocity planets have a measured radius")

### ✏️ Your turn 2

Find out, in `ps` and then in `pscomppars`, and print all of it:

1. For each of the two discovery methods that dominate the archive — `"Transit"` and
   `"Radial Velocity"` — how many planets `ps` holds, and how many of those have a radius.
   `column.count()` counts what is there and skips the holes.
2. The same two counts in `composite`. One of the four numbers will move enormously.
3. `pl_radeerr1` is the upper error bar on the radius. For the radial-velocity planets in each
   table, count how many have a radius **and no error bar at all**. A measurement without an
   uncertainty is not a measurement.
4. `value_counts()` on `pl_bmassprov` for the radial-velocity planets in `ps`.

Then answer it, in a line your code prints, on your own numbers: which radial-velocity planets can
you put on the same axis as the transit planets, and if you took every radius `composite` offers
for them instead, what exactly would you be comparing the transit radii against?

In [ ]:
# ← your answer here



Mass and radius really are related — a big planet is usually a heavy one — so estimating the second
from the first is not a silly thing to do, and somebody has to do something about a planet whose
radius nobody has measured. Whether such an estimate can then stand beside a measurement is a
separate question, and the way to settle it is to build the estimator yourself and look at what
comes out.

**Log axes:** When the values span factors of a thousand, plot the exponents instead and a curve becomes a line.
**Linear regression:** Draw the best straight line. Best means the smallest total miss.

### ✏️ Your turn 3

**Fit it.** Take every planet in `ps` whose `pl_bmassprov` is `"Mass"` — a real mass, not a floor —
and which also has a measured radius. Fit a straight line to `log10(radius)` against
`log10(mass)`. `np.log10(...)`, then `.reshape(-1, 1)` on the x values, then
`LinearRegression().fit(x, y)`. Print how many planets you fitted, the slope, the intercept and
`model.score(x, y)`, and draw the points and the line.

**Then use it.** Take the radial-velocity planets that have a mass but **no** measured radius, feed
their `pl_bmasse` — which for most of them is an `Msini`, a floor — through your line, and undo the
log with `10 ** ...`. Print the median radius your line invents for them.

**Then put three numbers side by side**: that invented median, the median measured radius of the
transit planets, and the median measured radius of the radial-velocity planets that *do* have one.

Now answer it, in a line your code prints, on those three numbers: "radial-velocity surveys find
planets several times larger than transit surveys do" is a claim you will read in print. On your
own output, is it a fact about planets or an output of the line you just fitted — and what would
you have to compare instead to settle it with measurements only?

In [ ]:
# ← your answer here



## Which telescope found them?

Three spacecraft found most of the transiting planets in this file, and they were built to do
different things.

**Kepler** (2009–2013) pointed at one 115-square-degree patch of Cygnus and did not move, staring
at 150,000 stars for four years. A long stare finds planets on long orbits, and a fixed field
reaches whatever stars happen to be in it, however faint and however far.

**K2** (2014–2018) was Kepler after two of its reaction wheels failed: the same telescope, but only
able to hold a field for about 80 days at a time, along the ecliptic.

**TESS** (2018–) does the opposite of Kepler. It sweeps almost the whole sky in 27-day sectors,
which is too short to catch a long orbit, and it was designed to look at stars that are **bright**
— which mostly means near — so that the planets it finds can be followed up from the ground.

Three instruments, three different slices of the same galaxy. The columns to ask with are
`disc_facility`, `pl_rade`, `pl_orbsmax` (the orbit's width in AU) and `sy_dist` (how far away the
system is, in parsecs).

### ✏️ Your turn 4

For the transit-discovered planets in `ps`, compare the three surveys. Their `disc_facility` strings
are exactly `"Kepler"`, `"K2"` and `"Transiting Exoplanet Survey Satellite (TESS)"` — print the
short name, not the long one, or your output will run off the page.

For each: how many planets, the median radius, the median orbit width in AU, the median system
distance in parsecs, and the fraction with a radius below 1.6 Earth radii. Then draw one figure
that puts Kepler's and TESS's radius distributions side by side — two `plt.hist` calls of
`np.log10` of the radius, with `plt.legend()`, is enough.

Now answer it, in a line your code prints, on your own numbers: the two big surveys disagree about
what a typical planet is. Is one of them wrong, and if not, what does it mean to ask which of the
two is a picture of the sky?

In [ ]:
# ← your answer here



## What would we have seen if geometry had not chosen for us?

Here is one thing a transit survey misses that you can compute exactly.

A planet transits only if its orbit is edge-on enough that it passes in front of the star from
where we sit. For orbits pointing in random directions, the chance of that is the star's radius
divided by the width of the orbit:

$$p_{\rm transit} \approx \frac{R_\star}{a}$$

Small numbers, and they follow from that one line. A planet like the Earth, one AU from a star like
the Sun, transits for about one observer in 215; a planet on a three-day orbit
round the same star transits for about one in 9. **Every close-in planet in the
archive stands for a handful of identical planets nobody could see; every distant one stands for
hundreds.** The bias is not subtle and it is not a matter of opinion — it is geometry.

The units do not match, so one conversion is needed. `st_rad` is in radii of the Sun and
`pl_orbsmax` is in AU. The IAU *defines* the nominal solar radius as 6.957e+08 m (2015
Resolution B3) and the astronomical unit as exactly 1.495978707e+11 m (2012 Resolution B2), so one
solar radius is 0.00465047 AU. Those are definitions, not measurements off a web page, which
is why they carry a resolution number instead of a read-date.

This estimate ignores the planet's own radius and any eccentricity. Both matter at the ten-percent
level; neither changes anything below.

### ✏️ Your turn 5

Work with the transit-discovered planets in `ps` that have `st_rad`, `pl_orbsmax` **and** `pl_rade`.
Print how many that is out of the 4,688 transit planets — the ones you have to drop are
themselves a selection, and it is worth knowing how big it is.

Compute the transit probability for each, using `st_rad * 0.00465047 / pl_orbsmax`. Print the
smallest, the median and the largest. Then plot `np.log10` of the probability against `np.log10` of
the orbit width.

**Use these names**: call the table `geometry` and the new column `p_transit`, because *Your turn 6*
carries straight on from them.

Now answer it, in a line your code prints, on your own numbers: for the median planet in this
sample, how many planets on similar orbits does each one you can see stand for — and does that
multiplier stay the same across the plot, or does it depend on where the planet is?

In [ ]:
# ← your answer here



The step every survey paper takes next is to treat `1 / p_transit` as a **weight**, so that a
planet standing in for many is counted many times. A share computed with those weights is an
estimate of the share in the *population*, rather than of the share in the file.

That is one correction, and it is the only bias in this dataset you can compute exactly.

### ✏️ Your turn 6

Give every planet in your `geometry` table the weight `1 / p_transit`, and compute the fraction
with a radius below 1.6 Earth radii **twice**: the plain fraction, and the weighted one, which
is `weights[small].sum() / weights.sum()` where `small` is the mask.

Do it for the whole transit sample **and** for Kepler alone, so you have four numbers. Draw them as
a bar chart, four bars, raw and weighted side by side for each sample.

Then, so you can see where the weight went, split `geometry` into four bands by orbit width with
`pd.qcut(geometry["pl_orbsmax"], 4)` and print, per band: how many planets, the median orbit width,
the fraction below 1.6 Earth radii, and the band's share of the total weight.

Now answer it, in a line your code prints, on your own numbers: correcting for geometry alone moves
the small-planet fraction by some amount and in some direction — say which, then use the four bands
to say *why* it moved that way, and what that tells you about the bias you have **not** corrected.

In [ ]:
# ← your answer here



## The question, answered

Mostly better telescopes. The typical known planet fell from 13.0 Earth radii to
2.47 because Kepler arrived, stared at one faint patch of sky for four years and
returned 2,750 planets with a median radius of
2.13; it has been climbing again because TESS arrived, swept the
bright nearby sky and returned 924 with a median of
3.22. Both censuses are honest and neither is the sky. What that
leaves unanswered is what the sky actually holds, and the next section is about how far this file
can get you towards it.

## What track T4 leans on

**The question.** Are we finding more Earth-like planets, or just better telescopes?

Nothing here is new. These are the weeks to look back at while you work, and the wording is the course's own.

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Catalogue completeness** | A catalogue lists what somebody's instruments recorded, not what happened. Where there are no seismometers there are no earthquakes in the file. |
| **NaN** | Where the file had nothing at all, pandas puts NaN. A NaN is a hole, not a zero. |
| **Table** | A table with a name on every column, so you ask for data by name instead of by position. |
| **Log axes** | When the values span factors of a thousand, plot the exponents instead and a curve becomes a line. |
| **Linear regression** | Draw the best straight line. Best means the smallest total miss. |
| **Bootstrap** | Ask the data the same question a thousand times, using a different random slice of itself each time. |
| **Confidence interval** | Not one number but the range your number would have wandered over, had the world rolled differently. |

### Code you will reach back for

| Function | What it does |
|---|---|
| `column.isna()` | a mask marking where the file had nothing |
| `column.value_counts()` | how often each value appears |
| `table.groupby(column)` | split the table into one group per value |
| `column.count() / column.median()` | how many, and the middle value |
| `np.log10(values)` | the exponent of every value at once — what turns a power law into a straight line |
| `array.reshape(-1, 1)` | one row per data point, which is the shape scikit-learn wants |
| `LinearRegression().fit(x, y)` | find the straight line with the smallest total miss |
| `model.coef_[0]` | the slope of the fitted line |
| `model.intercept_` | where the fitted line crosses zero |
| `model.predict(x)` | what the fitted line says y should be at each x |
| `model.score(x, y)` | R2 — the fraction of the up-and-down variation the line accounts for |
| `np.random.default_rng(seed)` | a random-number generator you can reproduce — the same seed gives the same draws |
| `np.random.choice(items, size=n, replace=True)` | pick n items at random, repeats allowed |
| `np.percentile(values, [2.5, 97.5])` | the two values that cut off the bottom and top 2.5% — a 95% interval |

## What your project must contain

Five sections, empty below, required of **every** EPS 88 project regardless of track. They are
headed here so the shape of a good answer is visible while you work. Fill them in as you go; they
are not a write-up you do at the end.

### ✏️ 1 · A one-sentence answer

Your claim and its uncertainty, in one sentence, at the top of your report. If you cannot put a
number and a range in it, you do not have a result yet.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 2 · The trivial baseline

Before any statistic, state the dumbest answer to your question and what it gives. Every later
number is reported against it.

On this track the baseline is one median over the whole file, with no year, no method and no
instrument in it: the archive's 4,742 measured radii have a middle value, and that
number is what somebody quoting "the typical exoplanet" is quoting. Say what it is, then say what
each later split bought you over it.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 3 · Split by structure

Earth data are correlated in space and in time, so whatever you split, resample or count as
independent has to be split along the structure that is really there — never at random across
rows.

This track fits one line and does not score it, so there is no train/test split to get wrong. The
same idea has teeth anyway, and the structure here is the **survey**: two planets found by the same
instrument in the same field share everything about how they were found. Name the unit you treated
as independent, say why, and say what changed when you grouped by it instead of by row.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 4 · What I got wrong

What failed, and what you believed before it failed. Honest failure is graded; a faked success is
not. Both of your *Predict before you run* guesses belong here if they were wrong.

*(Double-click this cell and replace this line with your answer.)*

### ✏️ 5 · AI disclosure

Which tool, what you asked it, what you changed in what it gave you, and how you checked that the
result was true.

*(Double-click this cell and replace this line with your answer.)*

## The open question

> **What does the REAL population look like?**

Nobody grading this knows the answer, and neither does the literature. Everything above is the
scaffolding; this is the project.

Here is what is actually established, and it is less than it looks. The fall from
13.0 Earth radii to 2.47 is Kepler arriving, and that is settled. That
each instrument returns a different galaxy is settled: Kepler's median planet sits
2.13 Earth radii at
771 parsecs, TESS's 3.22 at
115. What is **not** settled is the thing all of that is
evidence about: what the population is. Your turn 6 is where you found out what happens when you
correct one bias out of several, and it is the reason this question is open rather than merely
unfinished.

Four directions, none of them worked out here:

1. **Put the detections you never made back in.** Transit probability is one factor. The other is
   whether a transit that did occur was deep enough to find: the fraction of the star's light a
   planet blocks is `(pl_rade / st_rad)²` once both are in the same units, and one solar radius is
   109.076 Earth radii — the ratio of the two IAU 2015 Resolution B3 nominal radii
   quoted above. Compute that depth for every planet, find the value below which each survey stops
   reporting anything, and use that floor to say what each survey could not have seen. It is a
   completeness limit built from the archive's own columns, and it is the piece Your turn 6 was
   missing.
2. **Correct one survey properly rather than all of them badly.** Kepler is the only one here that
   stared at a fixed, documented set of stars, so it is the only one where "the sample" means
   something. Debias Kepler alone, quote the small-planet fraction with an interval, and say what
   population that number is a statement about — which stars, at what distances, on what orbits.
3. **Ask whether the population has structure the census is hiding.** The radius distribution of
   small planets is not smooth: there is a reported deficit near 1.8 Earth radii, and whether it is
   real or an artefact of the radii being uncertain has been argued about for years. Weighting
   changes a histogram's shape, not just its median — does the gap survive your weights, and does
   it survive in each survey separately?
4. **Decide what this file cannot answer.** The archive holds 1,612 planets with no
   radius at all and 958 radial-velocity planets whose mass is only a floor. Write
   down what fraction of the population you are estimating actually rests on a measurement, and
   what fraction rests on a model. If the honest answer is that most of it is model, that is a
   result.

And one that is bigger than a semester: what would a survey have to be like for its census to be a
population? Not "how many more planets" — what property would the sample need. Write down the
condition, then check it against the three surveys in this file, and see whether any of them could
ever have met it.

### ✏️ Your turn 7 — the first move

Before you close this notebook, answer this in a few sentences and then make the measurement in the
cell below your prose. Of everything left undone above, which **one** measurement would you make
first — what would it show if the population really is mostly small worlds, what would it show if
it is not, and what number would change your mind?

*(Double-click this cell and replace this line with your answer.)*

In [ ]:
# ← your answer here

